In [ ]:
import numpy as np
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import eigh

def compute_laplacian_spectra(a):
    degrees = np.sum(a, axis=1)
    d = np.diag(degrees)
    l_matrix = d - a
    eigenvalues, eigenvectors = eigh(l_matrix)
    return eigenvalues, eigenvectors[:, 1]

def build_transition_matrix(a):
    row_sums = a.sum(axis=1)
    row_sums[row_sums == 0] = 1
    return a / row_sums[:, np.newaxis]

def execute_stage_one():
    n = 150
    networks = {
        'Erdos-Renyi': nx.erdos_renyi_graph(n, 0.15, seed=42),
        'Barabasi-Albert': nx.barabasi_albert_graph(n, 4, seed=42),
        'Watts-Strogatz': nx.watts_strogatz_graph(n, 6, 0.1, seed=42)
    }

    records = []
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    for idx, (name, g) in enumerate(networks.items()):
        a = nx.to_numpy_array(g)
        l_eigenvalues, fiedler_vector = compute_laplacian_spectra(a)
        w = build_transition_matrix(a)
        w_eigenvalues = np.sort(np.abs(np.linalg.eigvals(w)))

        fiedler_value = l_eigenvalues[1]
        spectral_gap = 1.0 - w_eigenvalues[-2]
        algebraic_connectivity = fiedler_value / n

        records.append({
            "Topology": name,
            "Fiedler Value (λ_2)": f"{fiedler_value:.5f}",
            "Algebraic Conn.": f"{algebraic_connectivity:.5f}",
            "Spectral Gap (Δλ)": f"{spectral_gap:.5f}",
            "Max Fiedler Component": f"{np.max(np.abs(fiedler_vector)):.5f}"
        })

        axes[idx].hist(fiedler_vector, bins=30, color='black', alpha=0.7)
        axes[idx].set_title(f"{name}\nFiedler Vector Distribution")
        axes[idx].grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig("stage1_spectral_distribution.pdf", format="pdf", dpi=300)
    plt.close()

    df = pd.DataFrame(records)
    print(df.to_markdown(index=False, tablefmt="grid"))

if __name__ == '__main__':
    execute_stage_one()

+-----------------+-----------------------+-------------------+---------------------+-------------------------+
| Topology        |   Fiedler Value (λ_2) |   Algebraic Conn. |   Spectral Gap (Δλ) |   Max Fiedler Component |
+=================+=======================+===================+=====================+=========================+
| Erdos-Renyi     |              10.6835  |           0.07122 |             0.61104 |                 0.87123 |
+-----------------+-----------------------+-------------------+---------------------+-------------------------+
| Barabasi-Albert |               2.1934  |           0.01462 |             0.35386 |                 0.32077 |
+-----------------+-----------------------+-------------------+---------------------+-------------------------+
| Watts-Strogatz  |               0.30531 |           0.00204 |             0.05042 |                 0.20116 |
+-----------------+-----------------------+-------------------+---------------------+-------------------

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def generate_base_distribution(vocab_size):
    np.random.seed(42)
    p = np.random.dirichlet(np.ones(vocab_size))
    return p

def inject_semantic_noise(p, variance, steps):
    np.random.seed(42)
    distributions = [p]
    current_q = p.copy()
    for _ in range(steps):
        noise = np.random.normal(0, variance, len(p))
        current_q = np.abs(current_q + noise)
        current_q /= np.sum(current_q)
        distributions.append(current_q)
    return distributions

def compute_kl_divergence(p, q):
    epsilon = 1e-10
    q = np.clip(q, epsilon, 1.0)
    p = np.clip(p, epsilon, 1.0)
    return np.sum(p * np.log(p / q))

def execute_stage_two():
    vocab_size = 512
    steps = 50
    noise_profiles = [0.01, 0.05, 0.15]

    p = generate_base_distribution(vocab_size)
    records = []

    fig, ax = plt.subplots(figsize=(10, 6))

    for variance in noise_profiles:
        q_history = inject_semantic_noise(p, variance, steps)
        kl_history = [compute_kl_divergence(p, q) for q in q_history]

        final_kl = kl_history[-1]
        final_entropy = -np.sum(q_history[-1] * np.log(np.clip(q_history[-1], 1e-10, 1.0)))

        records.append({
            "Semantic Variance (σ²)": f"{variance:.4f}",
            "Initial Entropy (H)": f"{-np.sum(p * np.log(np.clip(p, 1e-10, 1.0))):.5f}",
            "Terminal Entropy (H)": f"{final_entropy:.5f}",
            "Asymptotic KL Div.": f"{final_kl:.5f}"
        })

        ax.plot(kl_history, label=f"Drift Variance = {variance}", linewidth=2)

    ax.set_title("Information Loss and Semantic Drift Trajectories")
    ax.set_xlabel("Communication Iterations")
    ax.set_ylabel("Kullback-Leibler Divergence")
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.savefig("stage2_semantic_divergence.pdf", format="pdf", dpi=300)
    plt.close()

    df = pd.DataFrame(records)
    print(df.to_markdown(index=False, tablefmt="grid"))

if __name__ == '__main__':
    execute_stage_two()

+--------------------------+-----------------------+------------------------+----------------------+
|   Semantic Variance (σ²) |   Initial Entropy (H) |   Terminal Entropy (H) |   Asymptotic KL Div. |
+==========================+=======================+========================+======================+
|                     0.01 |               5.81831 |                5.9472  |              0.91312 |
+--------------------------+-----------------------+------------------------+----------------------+
|                     0.05 |               5.81831 |                5.94564 |              0.88125 |
+--------------------------+-----------------------+------------------------+----------------------+
|                     0.15 |               5.81831 |                5.94745 |              0.86336 |
+--------------------------+-----------------------+------------------------+----------------------+


In [ ]:
import numpy as np
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

def build_normalized_matrix(n, topology_type):
    if topology_type == 'ER':
        g = nx.erdos_renyi_graph(n, 0.15, seed=42)
    elif topology_type == 'BA':
        g = nx.barabasi_albert_graph(n, 4, seed=42)
    elif topology_type == 'WS':
        g = nx.watts_strogatz_graph(n, 6, 0.1, seed=42)

    a = nx.to_numpy_array(g)
    row_sums = a.sum(axis=1)
    row_sums[row_sums == 0] = 1
    return a / row_sums[:, np.newaxis]

def simulate_system_collapse(w, gamma, semantic_noise, steps):
    np.random.seed(42)
    n = w.shape[0]
    x = np.random.rand(n)
    m = (1 - gamma) * np.eye(n) + gamma * w

    variance_trajectory = []

    for _ in range(steps):
        noise_vector = np.random.normal(0, semantic_noise, n)
        x = m @ x + noise_vector
        variance_trajectory.append(np.var(x))

    return variance_trajectory

def execute_stage_three():
    n = 150
    gamma = 0.90
    steps = 150
    critical_noise = 0.05

    topologies = ['ER', 'BA', 'WS']
    records = []

    fig, ax = plt.subplots(figsize=(10, 6))

    for topo in topologies:
        w = build_normalized_matrix(n, topo)
        var_traj = simulate_system_collapse(w, gamma, critical_noise, steps)

        asymptotic_variance = np.mean(var_traj[-20:])
        peak_variance = np.max(var_traj)
        convergence_stability = peak_variance / asymptotic_variance

        records.append({
            "Network Architecture": topo,
            "Critical Noise Bound": f"{critical_noise:.4f}",
            "Peak System Variance": f"{peak_variance:.5f}",
            "Asymptotic Variance": f"{asymptotic_variance:.5f}",
            "Instability Ratio (ρ)": f"{convergence_stability:.5f}"
        })

        ax.plot(var_traj, label=f"Architecture: {topo}", linewidth=2)

    ax.set_title("Multi-Agent Diagnostic Consensus Collapse under Critical Drift")
    ax.set_xlabel("Time Step (t)")
    ax.set_ylabel("Global State Variance (σ²)")
    ax.axhline(y=critical_noise, color='r', linestyle=':', label='Theoretical Noise Floor')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.savefig("stage3_consensus_cascade.pdf", format="pdf", dpi=300)
    plt.close()

    df = pd.DataFrame(records)
    print(df.to_markdown(index=False, tablefmt="grid"))

if __name__ == '__main__':
    execute_stage_three()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F
import numpy as np

def execute_stage_four():
    tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
    model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

    text = "Patient presents with acute myocardial infarction and elevated troponin levels."
    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    baseline_embedding = outputs.last_hidden_state[:, 0, :]

    variances_table = [0.1, 0.5, 1.0]
    records = []

    torch.manual_seed(42)
    for variance in variances_table:
        noise = torch.randn_like(baseline_embedding) * (variance ** 0.5)
        noisy_embedding = baseline_embedding + noise
        cos_sim = F.cosine_similarity(baseline_embedding, noisy_embedding).item()
        degradation = (1 - cos_sim) * 100

        records.append({
            "Noise Variance (σ²)": variance,
            "Terminal Cosine Sim": round(cos_sim, 4),
            "Semantic Degradation (%)": f"{round(degradation, 2)}%"
        })

    df = pd.DataFrame(records)
    print(df.to_markdown(index=False, tablefmt="grid"))

    variances_plot = np.linspace(0.0, 1.5, 100)
    cos_sims = []

    for v in variances_plot:
        noise = torch.randn_like(baseline_embedding) * (v ** 0.5)
        noisy_emb = baseline_embedding + noise
        sim = F.cosine_similarity(baseline_embedding, noisy_emb).item()
        cos_sims.append(sim)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(variances_plot, cos_sims, color='darkblue', linewidth=2)
    ax.set_title("Embedding Cosine Similarity Decay Under Variance Injection")
    ax.set_xlabel("Noise Variance (σ²)")
    ax.set_ylabel("Cosine Similarity to Baseline")
    ax.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.savefig("stage4_clinical_embedding_drift.pdf", format="pdf", dpi=300)
    plt.close()

if __name__ == '__main__':
    execute_stage_four()

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


+-----------------------+-----------------------+----------------------------+
|   Noise Variance (σ²) |   Terminal Cosine Sim | Semantic Degradation (%)   |
+=======================+=======================+============================+
|                   0.1 |                0.8493 | 15.07%                     |
+-----------------------+-----------------------+----------------------------+
|                   0.5 |                0.5939 | 40.61%                     |
+-----------------------+-----------------------+----------------------------+
|                   1   |                0.4671 | 53.29%                     |
+-----------------------+-----------------------+----------------------------+


In [ ]:
import numpy as np
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import eigh

def build_normalized_matrix(n, topology_type):
    if topology_type == 'ER':
        g = nx.erdos_renyi_graph(n, 0.15, seed=42)
    elif topology_type == 'BA':
        g = nx.barabasi_albert_graph(n, 4, seed=42)
    elif topology_type == 'WS':
        g = nx.watts_strogatz_graph(n, 6, 0.1, seed=42)
    a = nx.to_numpy_array(g)
    row_sums = a.sum(axis=1)
    row_sums[row_sums == 0] = 1
    return a / row_sums[:, np.newaxis]

def execute_stage_five():
    n = 150
    gamma = 0.90
    steps = 50
    noise_var = 0.01
    trials = 20

    topologies = [('Erdős-Rényi (ER)', 'ER'), ('Barabási-Albert (BA)', 'BA'), ('Watts-Strogatz (WS)', 'WS')]
    records = []

    fig3, ax3 = plt.subplots(figsize=(10, 6))
    fig2, ax2 = plt.subplots(figsize=(10, 6))
    fig1, axes1 = plt.subplots(1, 3, figsize=(15, 5))

    for idx, (name, topo) in enumerate(topologies):
        w = build_normalized_matrix(n, topo)
        m = (1 - gamma) * np.eye(n) + gamma * w

        eigenvalues, eigenvectors = eigh(m)
        principal_eigenvector = np.abs(eigenvectors[:, -1])
        sorted_eigenvector = np.sort(principal_eigenvector)[::-1]
        ax2.plot(sorted_eigenvector, label=name, linewidth=2)

        axes1[idx].imshow(m, cmap='viridis', aspect='auto')
        axes1[idx].set_title(f"{name}\nPhase Transition Matrix")

        all_trajectories = np.zeros((trials, steps + 1))

        for trial in range(trials):
            np.random.seed(42 + trial)
            x = np.random.normal(0, np.sqrt(noise_var), n)
            all_trajectories[trial, 0] = np.var(x)

            for t in range(1, steps + 1):
                noise_vector = np.random.normal(0, np.sqrt(noise_var), n)
                x = m @ x + noise_vector
                all_trajectories[trial, t] = np.var(x)

        mean_traj = np.mean(all_trajectories, axis=0)
        std_traj = np.std(all_trajectories, axis=0)

        ax3.plot(range(steps + 1), mean_traj, label=name, linewidth=2)
        ax3.fill_between(range(steps + 1), mean_traj - std_traj, mean_traj + std_traj, alpha=0.2)

        initial_var = mean_traj[0]
        midpoint_var = mean_traj[25]
        peak_var = np.max(mean_traj)
        amplification = peak_var / initial_var

        records.append({
            "Topology Architecture": name,
            "Initial (t=0)": f"{initial_var:.4f}",
            "Midpoint (t=25)": f"{midpoint_var:.4f}",
            "Peak (t=50)": f"{peak_var:.4f}",
            "Amplification (ρ)": f"{amplification:.4f}"
        })

    ax2.set_title("Tri-Topology Eigenvector Mass Concentration")
    ax2.set_xlabel("Node Rank (Sorted by Centrality)")
    ax2.set_ylabel("Eigenvector Mass Component")
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig("stage5_graph2_eigenvector_concentration.pdf", format="pdf", dpi=300)
    plt.close(fig2)

    fig1.tight_layout()
    fig1.savefig("stage5_graph1_phase_transition.pdf", format="pdf", dpi=300)
    plt.close(fig1)

    ax3.set_title("Tri-Topology Asymptotic Variance Trajectories")
    ax3.set_xlabel("Communication Hops (t)")
    ax3.set_ylabel("Global State Variance (σ²)")
    ax3.legend()
    ax3.grid(True, linestyle='--', alpha=0.6)
    fig3.tight_layout()
    fig3.savefig("stage5_graph3_variance_trajectories.pdf", format="pdf", dpi=300)
    plt.close(fig3)

    df = pd.DataFrame(records)
    print(df.to_markdown(index=False, tablefmt="grid"))

if __name__ == '__main__':
    execute_stage_five()

+-------------------------+-----------------+-------------------+---------------+---------------------+
| Topology Architecture   |   Initial (t=0) |   Midpoint (t=25) |   Peak (t=50) |   Amplification (ρ) |
+=========================+=================+===================+===============+=====================+
| Erdős-Rényi (ER)        |          0.0101 |            0.0103 |        0.0109 |              1.0766 |
+-------------------------+-----------------+-------------------+---------------+---------------------+
| Barabási-Albert (BA)    |          0.0101 |            0.0117 |        0.0124 |              1.2268 |
+-------------------------+-----------------+-------------------+---------------+---------------------+
| Watts-Strogatz (WS)     |          0.0101 |            0.0143 |        0.0153 |              1.5181 |
+-------------------------+-----------------+-------------------+---------------+---------------------+
